In [22]:
# Part 1
@show W = [randn(16, 2), randn(16, 16), randn(1, 16)]
@show b = [zeros(16), zeros(16), zeros(1)]
NN(x, y) = W[3]*tanh.(W[2]*tanh.(W[1]*[x, y]+b[1])+b[2])+b[3]

W = [randn(16, 2), randn(16, 16), randn(1, 16)] = [[1.3328472507174463 1.9552221140676602; -0.29895248978092936 -0.8694250764297771; -0.15470386727817223 0.2887282424972515; 0.7432854950035622 1.0440036819091207; 0.8634363680691368 0.9126572383516376; 0.4941346207799938 -0.32470665966823814; -1.7301778085875916 1.84021095987459; 0.7230315027816456 -0.894124260251975; 1.9715765476649052 -0.13378526203786195; 0.5010168448032775 -1.7620980896085858; -0.6490151882690034 1.3015418229781925; 0.7010279742011813 1.688588642893472; -0.13275034514984677 -1.5629793877095837; -0.939522518229798 -1.389708856607449; -1.4798754748107819 -0.19668711711588396; -0.7667350219577788 0.753282755313745], [-0.12034041454232536 -0.1693878741409059 -0.7620307130104944 -0.45560060907858546 -1.9815690931903391 -1.7325317975133363 2.0937088009028906 -0.9178547100090649 0.640186207594454 -0.6357135870214002 1.255339492477365 0.1771720043919772 -1.1892682119822209 -0.5970896076333481 -1.8471629501039362 0.204811730

NN (generic function with 1 method)

In [23]:
# Part 2
function NN_derivative(x, y, dx, dy)
    z1 = W[1] * [x, y] + b[1]
    z2 = W[2] * tanh.(z1) + b[2]
    W[3] * (sech.(z2) .^ 2 .* (W[2] * (sech.(z1) .^ 2 .* (W[1] * [dx, dy]))))
end


NN_derivative (generic function with 1 method)

In [24]:
# Part 3
function gradient_descent(f, df, u0, alpha; tol = 1e-10)
    un = u0
    step = df(un) * alpha
    epoch = 0
    while sqrt(sum(abs2, step)) >= tol
        epoch += 1
        println("Epoch: ", epoch, " Loss: ", f(un), " Step size: ", sqrt(sum(abs2, step)))
        un = un - step
        step = df(un) * alpha
    end
    [un, f(un)]
end

f(x) = sum(abs2, x)
df(x) = 2 .* x

gradient_descent(f, df, [1.0, 1.0], 0.01)


Epoch: 1 Loss: 2.0 Step size: 0.0282842712474619
Epoch: 2 Loss: 1.9207999999999998 Step size: 0.027718585822512663
Epoch: 3 Loss: 1.8447363200000002 Step size: 0.027164214106062408
Epoch: 4 Loss: 1.771684761728 Step size: 0.026620929823941163
Epoch: 5 Loss: 1.7015260451635712 Step size: 0.02608851122746234
Epoch: 6 Loss: 1.6341456137750938 Step size: 0.02556674100291309
Epoch: 7 Loss: 1.5694334474696001 Step size: 0.025055406182854834
Epoch: 8 Loss: 1.507283882949804 Step size: 0.024554298059197736
Epoch: 9 Loss: 1.4475954411849916 Step size: 0.024063212098013778
Epoch: 10 Loss: 1.390270661714066 Step size: 0.023581947856053505
Epoch: 11 Loss: 1.3352159435101891 Step size: 0.023110308898932435
Epoch: 12 Loss: 1.2823413921471856 Step size: 0.022648102720953783
Epoch: 13 Loss: 1.231560673018157 Step size: 0.02219514066653471
Epoch: 14 Loss: 1.1827908703666379 Step size: 0.021751237853204014
Epoch: 15 Loss: 1.1359523519001191 Step size: 0.021316213096139933
Epoch: 16 Loss: 1.0909686387648

2-element Vector{Any}:
  [3.482867618819059e-9, 3.482867618819059e-9]
 2.4260733700436688e-17

In [ ]:
# Part 4
using LinearAlgebra

nn_val(x, y) = first(NN(x, y))
nn_deriv_val(x, y, dx, dy) = first(NN_derivative(x, y, dx, dy))

const _h = 1e-6
nn_x(x, y) = nn_deriv_val(x, y, 1.0, 0.0)
nn_y(x, y) = nn_deriv_val(x, y, 0.0, 1.0)
nn_xx(x, y) = (nn_x(x + _h, y) - nn_x(x - _h, y)) / (2 * _h)
nn_yy(x, y) = (nn_y(x, y + _h) - nn_y(x, y - _h)) / (2 * _h)

g(x, y) = x * (1 - x) * y * (1 - y)
g_x(x, y) = y * (1 - y) * (1 - 2x)
g_xx(x, y) = -2 * y * (1 - y)
g_y(x, y) = x * (1 - x) * (1 - 2y)
g_yy(x, y) = -2 * x * (1 - x)

function u_pred(x, y)
    n = nn_val(x, y)
    n_x = nn_x(x, y)
    n_y = nn_y(x, y)
    n_xx = nn_xx(x, y)
    n_yy = nn_yy(x, y)
    g_ = g(x, y)
    gx_ = g_x(x, y)
    gy_ = g_y(x, y)
    gxx_ = g_xx(x, y)
    gyy_ = g_yy(x, y)
    u_xx = gxx_ * n + 2 * gx_ * n_x + g_ * n_xx
    u_yy = gyy_ * n + 2 * gy_ * n_y + g_ * n_yy
    (; u_xx, u_yy)
end

const n_colloc = 15
const _xs = [(i / (n_colloc + 1)) for i = 1:n_colloc]
const _ys = [(j / (n_colloc + 1)) for j = 1:n_colloc]

function poisson_loss_at(u_params)
    unpack!(u_params)
    residual_sq = 0.0
    for x in _xs
        for y in _ys
            (; u_xx, u_yy) = u_pred(x, y)
            rhs = sin(pi * x) * sin(pi * y)
            residual = u_xx + u_yy + rhs
            residual_sq += residual^2
        end
    end
    residual_sq / (n_colloc * n_colloc)
end

function pack!(W, b)
    vcat(vec(W[1]), vec(W[2]), vec(W[3]), vec(b[1]), vec(b[2]), vec(b[3]))
end
function unpack!(v)
    i = 0
    W[1][:] = v[(i+1):(i+32)];
    i += 32
    W[2][:] = v[(i+1):(i+256)];
    i += 256
    W[3][:] = v[(i+1):(i+16)];
    i += 16
    b[1][:] = v[(i+1):(i+16)];
    i += 16
    b[2][:] = v[(i+1):(i+16)];
    i += 16
    b[3][:] = v[(i+1):(i+1)];
    i += 1
    nothing
end

const _eps = 1e-7
function poisson_loss_grad(u_params)
    grad = similar(u_params)
    f0 = poisson_loss_at(u_params)
    for i in eachindex(u_params)
        u_plus = copy(u_params);
        u_plus[i] += _eps
        u_minus = copy(u_params);
        u_minus[i] -= _eps
        grad[i] = (poisson_loss_at(u_plus) - poisson_loss_at(u_minus)) / (2 * _eps)
    end
    grad
end

u0 = pack!(W, b)

alpha = 5e-1
result = gradient_descent(poisson_loss_at, poisson_loss_grad, u0, alpha; tol = 5e-3)
u_opt, loss_opt = result
unpack!(u_opt)
println("Final loss: ", loss_opt)

u_exact(x, y) = sin(pi * x) * sin(pi * y) / (pi^2)
u_approx(x, y) = g(x, y) * nn_val(x, y)
println("At (0.5, 0.5): u_exact = ", u_exact(0.5, 0.5), ", u_approx = ", u_approx(0.5, 0.5))

Epoch: 1 Loss: 1.03611921695084 Step size: 0.4488300423193537
Epoch: 2 Loss: 1.3225856075381384 Step size: 0.8085320805641513
Epoch: 3 Loss: 1.5242219933114982 Step size: 0.6028523672083645
Epoch: 4 Loss: 0.5460426737224894 Step size: 0.3365739421223051
Epoch: 5 Loss: 0.43106697479324313 Step size: 0.35491570324016614
Epoch: 6 Loss: 0.3899660318304018 Step size: 0.3122545303009668
Epoch: 7 Loss: 0.2488792103574738 Step size: 0.24089022304192784
Epoch: 8 Loss: 0.16904155004362534 Step size: 0.18518566418269883
Epoch: 9 Loss: 0.11889958370350254 Step size: 0.14172066351186258
Epoch: 10 Loss: 0.08919580914690294 Step size: 0.10860537786228477
Epoch: 11 Loss: 0.07000862991524508 Step size: 0.08302380492008452
Epoch: 12 Loss: 0.05809998497076078 Step size: 0.0638068464033925
Epoch: 13 Loss: 0.0501450482407068 Step size: 0.0492247475538248
Epoch: 14 Loss: 0.044783268236070546 Step size: 0.03835983646978989
Epoch: 15 Loss: 0.0409203692360591 Step size: 0.030244179376527538
Epoch: 16 Loss: 0.0